In [ ]:
# Fahmida Kabir 
# Last Updated: 01/05/2025
# This specific file works on the facial recognition aspect for the robot canine, there are a lot of goals that the facial recognition will have to do, due to the time and speciality needed YOLO will be used.

In [ ]:
# Installing libraries required for the YOLO 5 to work 

#!pip install torch torchvision torchaudio
#!pip install opencv-python pandas
# pip install backports.tarfile


In [ ]:
# Libraries that are needed
import os
import sys
import time
import math
import cv2
import numpy as np
import torch
import torch.nn as nn
import random
from collections import deque
import pickle
import threading
import queue
# import RPi.GPIO as GPIO
from pathlib import Path
from gpiozero import Buzzer
import smbus


ModuleNotFoundError: No module named 'RPi'

In [ ]:
# Set up GPIO for the buzzer
BUZZER_PIN = 17
GPIO.setmode(GPIO.BCM)
GPIO.setup(BUZZER_PIN, GPIO.OUT)

# Initialize Freenove Robot Dog
robot_dog = FreenoveRobotDog()

# Load YOLOv5 model
model = torch.hub.load('ultralytics/yolov5', 'yolov5n', device='cpu')  # 'yolov5n' is the smallest model
model.conf = 0.5  # Confidence threshold

# OpenCV Video Capture on default camera [1]
cap = cv2.VideoCapture(0)

# Initialize face tracking
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

# Dummy RL class for path optimization (you can integrate your RL model here)
class path_optimization:
    def __init__(self):
        self.state = [0, 0]
        self.goal = [0, 0]
    
    def update_goal(self, ball_position):
        self.goal = ball_position
    
    def choose_action(self):
        # Choose action based on state and goal (this should be an RL decision)
        # For now, we simulate the robot moving towards the ball randomly
        return random.choice([1, -1])  # Random action (move forward or backward)
    
    def move_towards_goal(self):
        # For simplicity, simulate moving towards goal with random step
        dx = self.goal[0] - self.state[0]
        dy = self.goal[1] - self.state[1]
        action = None

        if abs(dx) > abs(dy):
            action = "forward" if dx > 0 else "backward"
        else:
            action = "left" if dy < 0 else "right"

        # Send the action command to the Freenove robot
        if action == "forward":
            robot_dog.move_forward()
        elif action == "backward":
            robot_dog.move_backward()
        elif action == "left":
            robot_dog.turn_left()
        elif action == "right":
            robot_dog.turn_right()
        
        self.state[0] += np.sign(dx)
        self.state[1] += np.sign(dy)
        return self.state

# Initialize RL path optimizer
path_optimizer = path_optimization()

# Red ball detection function
def detect_red_ball(frame):
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    # Red color range
    lower_red1 = np.array([0, 120, 70])
    upper_red1 = np.array([10, 255, 255])
    lower_red2 = np.array([170, 120, 70])
    upper_red2 = np.array([180, 255, 255])

    mask1 = cv2.inRange(hsv, lower_red1, upper_red1)
    mask2 = cv2.inRange(hsv, lower_red2, upper_red2)
    mask = mask1 | mask2

    contours, _ = cv2.findContours(mask, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)
    if contours:
        c = max(contours, key=cv2.contourArea)
        if cv2.contourArea(c) > 500:
            x, y, w, h = cv2.boundingRect(c)
            return (x, y, w, h)
    return None

# Function to buzz when red ball is found
def buzz():
    for _ in range(3):
        GPIO.output(BUZZER_PIN, GPIO.HIGH)
        time.sleep(0.1)
        GPIO.output(BUZZER_PIN, GPIO.LOW)
        time.sleep(0.1)

# Face recognition function
def recognize_faces(frame):
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, 1.1, 4)
    face_labels = []
    
    for idx, (x, y, w, h) in enumerate(faces):
        label = f"Person {chr(65 + idx)}"  # Label faces as Person A, B, C...
        cv2.rectangle(frame, (x, y), (x+w, y+h), (0, 255, 0), 2)
        cv2.putText(frame, label, (x, y-10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)
        face_labels.append(label)
    
    return face_labels

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # Object & face detection using YOLOv5
    results = model(frame)
    detections = results.pandas().xyxy[0]

    # Process detections
    for _, det in detections.iterrows():
        label = det['name']
        conf = det['confidence']
        x1, y1, x2, y2 = map(int, [det['xmin'], det['ymin'], det['xmax'], det['ymax']])
        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(frame, f"{label} {conf:.2f}", (x1, y1-10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)

        if label == 'person':
            # If a face is detected, label it (this will be handled by the face detection as well)
            recognize_faces(frame)

    # Red ball detection and tracking
    red_box = detect_red_ball(frame)
    if red_box:
        x, y, w, h = red_box
        cv2.rectangle(frame, (x, y), (x+w, y+h), (0, 0, 255), 2)
        cv2.putText(frame, "Red Ball", (x, y-10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 1)
        
        # Buzz when the ball is found
        buzz()

        # Update path optimizer with red ball position
        ball_position = [x + w // 2, y + h // 2]  # Ball's center
        path_optimizer.update_goal(ball_position)
    
    # Move the robot towards the ball (simplified, integrate RL here)
    robot_position = path_optimizer.move_towards_goal()

    # Display the frame
    cv2.imshow("Robot Dog Vision", frame)

    # Stop on 'q' press
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
GPIO.cleanup()


In [ ]:
# References 
# [1] [1]GeeksforGeeks, “Python OpenCV: Capture Video from Camera,” GeeksforGeeks, Jan. 26, 2020. https://www.geeksforgeeks.org/python-opencv-capture-video-from-camera/ (accessed May 01, 2025).